# AI Powered E-Commerce Customer Intelligence System
Junior Data Scientist – Final Hackathon

**Database:** ecommerce_hackathon.db (SQLite3, 4 tables, 100,000 rows)

**Goal:** Inspect and clean the raw database, run SQL business analysis, build churn (ML + DL)
and review sentiment (NLP) models, and deploy a Streamlit app.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3 as sql
import warnings
warnings.filterwarnings('ignore')

In [14]:
conn = sql.connect('ecommerce_hackathon.db')

In [13]:
# List tables
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)
print(tables)

        name
0  customers
1   products
2     orders
3    reviews


In [16]:
# Load the four tables into DataFrames
customers = pd.read_sql("SELECT * FROM customers", conn)
products = pd.read_sql("SELECT * FROM products", conn)
orders = pd.read_sql("SELECT * FROM orders", conn)
reviews = pd.read_sql("SELECT * FROM reviews", conn)

In [18]:
table_names = tables["name"].tolist()

for table_name in table_names:
    df = pd.read_sql(f"SELECT * FROM {table_name}",conn)
    
    print(f"\n{'='*50}")
    print(f"TABLE: {table_name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    


TABLE: customers
Shape: (8000, 9)
Columns: ['customer_id', 'customer_name', 'age', 'gender', 'city', 'signup_date', 'membership_type', 'email', 'phone']

TABLE: products
Shape: (1000, 6)
Columns: ['product_id', 'product_name', 'category', 'brand', 'unit_price', 'stock_quantity']

TABLE: orders
Shape: (65000, 10)
Columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'returned']

TABLE: reviews
Shape: (26000, 6)
Columns: ['review_id', 'customer_id', 'product_id', 'review_date', 'rating', 'review_text']


In [21]:
def inspect_table(df, table_name):
    print(f"\n{'='*60}")
    print(f"TABLE: {table_name}")
    print(f"{'='*60}")

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nUnique Values:")
    print(df.nunique())

    

In [22]:
for table_name in table_names:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    inspect_table(df, table_name)


TABLE: customers
Shape: (8000, 9)

Columns:
['customer_id', 'customer_name', 'age', 'gender', 'city', 'signup_date', 'membership_type', 'email', 'phone']

Data Types:
customer_id          int64
customer_name          str
age                float64
gender                 str
city                   str
signup_date            str
membership_type        str
email                  str
phone                  str
dtype: object

Missing Values:
customer_id          0
customer_name        0
age                100
gender               0
city                 0
signup_date          0
membership_type      0
email                0
phone                0
dtype: int64

Duplicate Rows:
0

Unique Values:
customer_id        8000
customer_name       800
age                  46
gender                2
city                 44
signup_date        1628
membership_type       4
email              7965
phone              8000
dtype: int64

TABLE: products
Shape: (1000, 6)

Columns:
['product_id', 'product_name',

## Task A: Database Inspection and Cleaning

The four tables are already loaded. Shapes and columns were printed above. The next cells look for missing values, duplicates, invalid numbers, messy categories, and bad dates, then apply cleaning that keeps as many rows as possible.

In [ ]:
# Targeted data-quality checks (beyond shape / null counts)
print("CUSTOMERS")
print("  missing age:", customers["age"].isna().sum())
print("  duplicate emails:", customers["email"].duplicated().sum())
print("  city variants:", customers["city"].nunique())
print("  gender values:", customers["gender"].unique().tolist())
print("  membership values:", customers["membership_type"].unique().tolist())
print("  invalid signup_date:", pd.to_datetime(customers["signup_date"], errors="coerce").isna().sum())

print("\nPRODUCTS")
print("  missing brand:", products["brand"].isna().sum())
print("  category variants:", products["category"].nunique(), products["category"].unique().tolist())
print("  unit_price <= 0:", (products["unit_price"] <= 0).sum())
print("  stock < 0:", (products["stock_quantity"] < 0).sum())

print("\nORDERS")
print("  missing payment_method:", orders["payment_method"].isna().sum())
print("  missing delivery_days:", orders["delivery_days"].isna().sum())
print("  quantity == 0:", (orders["quantity"] == 0).sum())
print("  unit_price < 0:", (orders["unit_price"] < 0).sum())
print("  invalid order_date:", pd.to_datetime(orders["order_date"], errors="coerce").isna().sum())
print("  payment methods:", orders["payment_method"].value_counts(dropna=False).to_dict())
print("  orphan customer_id:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("  orphan product_id:", (~orders["product_id"].isin(products["product_id"])).sum())

print("\nREVIEWS")
print("  missing review_text:", reviews["review_text"].isna().sum())
print("  blank review_text:", (reviews["review_text"].astype(str).str.strip() == "").sum())
print("  rating value counts:\n", reviews["rating"].value_counts().sort_index().to_string())
print("  invalid review_date:", pd.to_datetime(reviews["review_date"], errors="coerce").isna().sum())
print("  orphan customer_id:", (~reviews["customer_id"].isin(customers["customer_id"])).sum())
print("  orphan product_id:", (~reviews["product_id"].isin(products["product_id"])).sum())

### Data quality summary

| Table | Shape | Important issues |
|---|---|---|
| **customers** | 8,000 × 9 | 100 missing ages (~1.3%). 35 duplicate emails assigned to different people. 44 city strings for 13 real cities (case and extra spaces). No row duplicates. Signup dates are valid (2022-01-01 to 2026-06-30). |
| **products** | 1,000 × 6 | 20 missing brands. 19 category labels for 8 real categories (case/whitespace). Prices and stock are valid. |
| **orders** | 65,000 × 10 | 55 missing payment methods, 40 missing delivery days. 45 orders with `quantity = 0`. 35 negative `unit_price` values (absolute values match the product catalog — sign errors). 30 invalid dates (`unknown`, `2025-00-14`, `31-02-2025`, `2026/13/01`). No broken foreign keys. |
| **reviews** | 26,000 × 6 | 48 null + 42 blank review texts. 11 ratings of 0 and 14 ratings of 6 (outside 1–5). 16 `unknown` review dates. |

No full-row duplicates. All order/review IDs point at real customers and products, so I will **fix values instead of dropping rows** unless a field is unusable for a later task (blank review text is kept, then handled in NLP).

In [ ]:
# Work on copies so the raw extracts stay available for comparison
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
reviews_clean = reviews.copy()

# --- customers ---
# Same city, different spelling/spacing ("KARACHI", " Karachi ") would split every city-level metric.
customers_clean["city"] = customers_clean["city"].str.strip().str.title()

# Age is required later for churn. 100 gaps are small and ages look realistic (18-65),
# so fill with the median age of the same gender instead of dropping customers.
customers_clean["age"] = customers_clean.groupby("gender")["age"].transform(
    lambda s: s.fillna(s.median())
)
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"])

# Duplicate emails belong to different customer_ids / names / phones.
# Dropping them would delete real customers and break order foreign keys, so they are kept.

# --- products ---
products_clean["category"] = (
    products_clean["category"].str.strip().str.title()
    .str.replace(r"\s+", " ", regex=True)
)

def brand_from_name(name):
    name = str(name)
    if name.startswith("Saeed Ghani"):
        return "Saeed Ghani"
    if name.startswith("The Ordinary"):
        return "The Ordinary"
    return name.split()[0]

# Missing brands still appear as the first words of product_name.
products_clean["brand"] = products_clean["brand"].fillna(
    products_clean["product_name"].map(brand_from_name)
)

# --- orders ---
# Negative prices are almost exactly -1 * catalog price: treat as a sign typo, not a refund.
orders_clean["unit_price"] = orders_clean["unit_price"].abs()

# A completed order with quantity 0 cannot contribute revenue; the typical order quantity is 1.
orders_clean.loc[orders_clean["quantity"] == 0, "quantity"] = 1

# Payment and delivery gaps are small. Mode/median keep the row and avoid biasing rare methods.
orders_clean["payment_method"] = orders_clean["payment_method"].fillna(
    orders_clean["payment_method"].mode()[0]
)
orders_clean["delivery_days"] = orders_clean["delivery_days"].fillna(
    orders_clean["delivery_days"].median()
)

orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
orders_clean["order_date"] = orders_clean["order_date"].fillna(
    orders_clean["order_date"].median()
)

# --- reviews ---
reviews_clean["rating"] = reviews_clean["rating"].clip(lower=1, upper=5)
reviews_clean["review_text"] = reviews_clean["review_text"].fillna("").str.strip()
reviews_clean["review_date"] = pd.to_datetime(reviews_clean["review_date"], errors="coerce")
reviews_clean["review_date"] = reviews_clean["review_date"].fillna(
    reviews_clean["review_date"].median()
)

# Persist cleaned tables for SQL (Task B) and Streamlit without overwriting the raw database.
clean_conn = sql.connect("ecommerce_hackathon_clean.db")
customers_clean.to_sql("customers", clean_conn, if_exists="replace", index=False)
products_clean.to_sql("products", clean_conn, if_exists="replace", index=False)
orders_clean.to_sql("orders", clean_conn, if_exists="replace", index=False)
reviews_clean.to_sql("reviews", clean_conn, if_exists="replace", index=False)
clean_conn.close()
print("Saved cleaned tables to ecommerce_hackathon_clean.db")

In [ ]:
# After-cleaning checks
print("customers age missing:", customers_clean["age"].isna().sum(),
      "| cities:", sorted(customers_clean["city"].unique()))
print("products brand missing:", products_clean["brand"].isna().sum(),
      "| categories:", sorted(products_clean["category"].unique()))
print("orders qty==0:", (orders_clean["quantity"] == 0).sum(),
      "| negative price:", (orders_clean["unit_price"] < 0).sum(),
      "| pay NA:", orders_clean["payment_method"].isna().sum(),
      "| delivery NA:", orders_clean["delivery_days"].isna().sum(),
      "| bad dates:", orders_clean["order_date"].isna().sum())
print("reviews rating range:", reviews_clean["rating"].min(), "-", reviews_clean["rating"].max(),
      "| text NA:", reviews_clean["review_text"].isna().sum(),
      "| blank text:", (reviews_clean["review_text"] == "").sum(),
      "| bad dates:", reviews_clean["review_date"].isna().sum())
print("row counts unchanged:",
      customers_clean.shape, products_clean.shape, orders_clean.shape, reviews_clean.shape)

### Why these cleaning decisions

- **Did not drop rows.** The brief says not to delete problematic records without a reason. Issues were sparse, and customers/orders are linked, so fixing values keeps the 100,000-row structure intact.
- **Age filled by gender-median.** Needed for the churn model. A typical adult age in the same gender group is more honest than dropping 100 customers or filling 0.
- **City and category standardized (strip + title case).** Otherwise Karachi vs KARACHI vs ` Karachi ` would look like different markets/categories in SQL and charts.
- **Brand inferred from product name.** Every missing brand is already in the product title (`Samsung Power Bank Pro` → Samsung). Two-word brands (`Saeed Ghani`, `The Ordinary`) are handled explicitly.
- **Negative prices → absolute value.** They match catalog prices with the sign flipped, so they are typos, not refunds. Leaving them would understate revenue.
- **Quantity 0 → 1.** These rows still have a price and payment method; a sold quantity of 0 is not a real transaction. Mode quantity is already 1.
- **Missing payment/delivery filled with mode/median.** Only 55 and 40 rows. Imputing keeps revenue and delivery-time features complete for churn.
- **Broken dates → median valid date.** Strings like `31-02-2025` cannot be parsed. 30/65,000 orders is too few to invent a custom calendar, and too many to leave as NaT if we group by month.
- **Ratings clipped to 1–5.** Task F labels sentiment from 1–2 / 3 / 4–5. 0 and 6 are out of scale, so they are treated as the nearest valid score.
- **Blank reviews kept as empty strings.** They stay in the table for counts; the sentiment task will skip unusable text instead of deleting review IDs now.
- **Duplicate emails kept.** Different names, phones, and IDs share an email — that is a data bug, not a duplicate person. Deleting them would drop orders.
- **Raw DB left untouched.** Cleaned tables go to `ecommerce_hackathon_clean.db` so Task B SQL and the Streamlit app run on analysis-ready data.

## Task B: SQL Business Analysis

**What:** five business questions in SQL (not pandas groupby).  
**Why:** the brief requires SQL against SQLite. We run the queries in `business_queries.sql` on the **cleaned** database so negative prices and messy cities do not distort revenue.

In [ ]:
# Task B: execute the five required queries from the SQL file on the cleaned DB.
import re

sql_conn = sql.connect("ecommerce_hackathon_clean.db")
sql_text = open("business_queries.sql", encoding="utf-8").read()
queries = [q.strip() for q in re.split(r";\s*\n", sql_text) if q.strip() and not q.strip().startswith("--")]
# Keep only SELECT statements (file also has comment-only chunks).
selects = [q for q in queries if q.lstrip().upper().startswith("SELECT") or "SELECT" in q.upper()]
# Split more reliably on semicolons after stripping comments.
chunks = []
buf = []
for line in sql_text.splitlines():
    if line.strip().startswith("--"):
        continue
    buf.append(line)
    if ";" in line:
        q = "\n".join(buf).strip().rstrip(";").strip()
        buf = []
        if q:
            chunks.append(q)

titles = [
    "1. Total net revenue",
    "2. Top 10 customers by spending",
    "3. Category-wise revenue / orders / quantity",
    "4. Monthly net revenue trend",
    "5. Top 5 products by net revenue",
]
for title, q in zip(titles, chunks):
    print("\n" + title)
    print("-" * 40)
    display(pd.read_sql(q, sql_conn))
sql_conn.close()

## Task C: Exploratory Data Analysis

**What:** four charts on cleaned data. **Why cleaned:** messy city/category labels would split the bars. Net revenue = `quantity × unit_price × (1 − discount)`.

In [ ]:
# Task C: four required charts. One figure so we can talk through them in order during the presentation.
orders_eda = orders_clean.merge(
    products_clean[["product_id", "category"]], on="product_id"
).merge(
    customers_clean[["customer_id", "city"]], on="customer_id"
)
orders_eda["net_revenue"] = (
    orders_eda["quantity"] * orders_eda["unit_price"] * (1 - orders_eda["discount"])
)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

monthly = orders_eda.groupby(orders_eda["order_date"].dt.to_period("M"))["net_revenue"].sum()
monthly.index = monthly.index.to_timestamp()
axes[0, 0].plot(monthly.index, monthly.values / 1e6, color="#1f4e79")
axes[0, 0].set_title("Monthly net revenue")
axes[0, 0].set_ylabel("PKR millions")

cat = orders_eda.groupby("category")["net_revenue"].sum().sort_values()
axes[0, 1].barh(cat.index, cat.values / 1e6, color="#2e75b6")
axes[0, 1].set_title("Category net revenue")
axes[0, 1].set_xlabel("PKR millions")

city = orders_eda.groupby("city")["net_revenue"].sum().sort_values()
axes[1, 0].barh(city.index, city.values / 1e6, color="#548235")
axes[1, 0].set_title("City net revenue")
axes[1, 0].set_xlabel("PKR millions")

ret = orders_eda.groupby("category")["returned"].mean().mul(100).sort_values()
axes[1, 1].barh(ret.index, ret.values, color="#c45911")
axes[1, 1].set_title("Return rate by category")
axes[1, 1].set_xlabel("Return rate (%)")

plt.tight_layout()
plt.show()

### Business insights

1. **July 2026 is the peak (~PKR 83.6M), August drops ~12%.** Plan stock and riders for a mid-year spike, not a straight line of growth.
2. **Electronics is ~70% of revenue; Fashion has more orders but the worst return rate (~11%).** Protect Electronics supply. Fashion returns are a process problem; Electronics returns still cost more rupees because AOV is high (~PKR 57k).
3. **Karachi + Lahore + Islamabad ≈ 54% of sales.** Put faster delivery and ads there first. Smaller cities still buy often per customer — expand later, not with a warehouse everywhere.

## Task D: Customer Churn (Machine Learning)

**What we are doing:** predict which existing customers will **not buy again** in Jun–Aug 2026.

**Why this label:** the brief fixes the dates so we do not leak the future into the features.

| Piece | Rule |
|---|---|
| Who is scored | Customers with at least one order **on or before 31 May 2026** |
| Features | Built **only** from those history orders + age / membership |
| Target window | 1 Jun 2026 – 31 Aug 2026 |
| `churn = 1` | No order in the target window |
| `churn = 0` | At least one order in the target window |

**Leakage check:** June–August orders are used **only** for the label, never as `total_orders` / spending / recency.

In [ ]:
# Task D: build the churn table. History <= 31 May 2026; label from Jun-Aug only.
FEATURE_END = pd.Timestamp("2026-05-31")
TARGET_START = pd.Timestamp("2026-06-01")
TARGET_END = pd.Timestamp("2026-08-31")

orders_ml = orders_clean.copy()
orders_ml["net_revenue"] = (
    orders_ml["quantity"] * orders_ml["unit_price"] * (1 - orders_ml["discount"])
)

history = orders_ml[orders_ml["order_date"] <= FEATURE_END]
target = orders_ml[
    (orders_ml["order_date"] >= TARGET_START) & (orders_ml["order_date"] <= TARGET_END)
]
active_in_window = set(target["customer_id"])

# Required features from purchase history (snapshot date = 31 May 2026).
churn_df = history.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    total_spending=("net_revenue", "sum"),
    avg_order_value=("net_revenue", "mean"),
    last_order_date=("order_date", "max"),
    return_rate=("returned", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
).reset_index()

churn_df["days_since_last_order"] = (FEATURE_END - churn_df["last_order_date"]).dt.days
churn_df = churn_df.drop(columns="last_order_date")

# Label: 1 = silent in Jun-Aug (churned), 0 = still buying.
churn_df["churn"] = (~churn_df["customer_id"].isin(active_in_window)).astype(int)

churn_df = churn_df.merge(
    customers_clean[["customer_id", "age", "membership_type"]],
    on="customer_id",
    how="left",
)

print("Eligible customers (bought by 31 May 2026):", len(churn_df))
print("History orders used for features:", len(history))
print("Target-window orders (label only):", len(target))
print("Churn rate:\n", churn_df["churn"].value_counts(normalize=True).round(3))

In [ ]:
# Train/test split first, then fit. Same split for both models so the comparison is fair.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report,
)

feature_cols = [
    "total_orders", "total_spending", "avg_order_value",
    "days_since_last_order", "return_rate", "avg_delivery_days",
    "age", "membership_type",
]
num_cols = [c for c in feature_cols if c != "membership_type"]
cat_cols = ["membership_type"]

X = churn_df[feature_cols]
y = churn_df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape, "train churn rate:", round(y_train.mean(), 3))

# LR needs scaled numbers; RF does not. One-hot membership so both models see the same columns.
pre_lr = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])
pre_rf = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])

log_reg = Pipeline([
    ("pre", pre_lr),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
rand_forest = Pipeline([
    ("pre", pre_rf),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        random_state=42, n_jobs=-1,
    )),
])

log_reg.fit(X_train, y_train)
rand_forest.fit(X_train, y_train)
print("Both models trained.")

In [ ]:
# Evaluate on the same test set. Churn=1 is the class we care about catching (recall).
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(classification_report(y_test, y_pred, digits=3, target_names=["active (0)", "churn (1)"]))
    print("ROC-AUC:", round(metrics["roc_auc"], 3))
    return metrics, y_pred, y_proba

lr_metrics, lr_pred, lr_proba = evaluate_model("Logistic Regression", log_reg, X_test, y_test)
rf_metrics, rf_pred, rf_proba = evaluate_model("Random Forest", rand_forest, X_test, y_test)

compare = pd.DataFrame([lr_metrics, rf_metrics]).set_index("model").round(3)
print("\nSide-by-side comparison (positive class = churn)")
display(compare)

In [ ]:
# Confusion matrices and ROC curves (required). Accuracy alone hides whether we catch churners.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in zip(axes, ["Logistic Regression", "Random Forest"], [lr_pred, rf_pred]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Active", "Churn"], yticklabels=["Active", "Churn"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for name, proba in [("Logistic Regression", lr_proba), ("Random Forest", rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curve")
ax.legend()
plt.tight_layout()
plt.show()

### Which churn model we use, and why

Classes are almost balanced (~49% churn), so **accuracy ~70% is not enough on its own**. We care about **recall** (catch customers who will go quiet) and **ROC-AUC** (ranking who to call first).

**We put Logistic Regression in the Streamlit app.** In our test split it matches Random Forest on F1 and is slightly stronger on ROC-AUC, it is faster, and we can explain coefficients to management. Random Forest is a close backup if we later add more non-linear features.

In [ ]:
# Save the pipelines with relative paths so Streamlit can load them (rule 6).
import joblib, os
os.makedirs("models", exist_ok=True)
joblib.dump(log_reg, "models/churn_logreg.joblib")
joblib.dump(rand_forest, "models/churn_rf.joblib")
print("Saved churn models to models/")

## Task E: Deep Learning (same churn data)

**What:** a small feed-forward net `32 → 16 → 1` on the **same train/test split**.  
**Why this size:** the brief asks for a compact network, not a large model. Inputs are already scaled (same preprocessor as Logistic Regression). Hidden layers use ReLU; the output is a probability (sigmoid / log-loss).

In [ ]:
# Task E: 32 -> 16 -> 1 MLP. Same X_test as LR and RF so the comparison is fair.
from sklearn.neural_network import MLPClassifier

mlp = Pipeline([
    ("pre", pre_lr),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        max_iter=500,
        random_state=42,
    )),
])
mlp.fit(X_train, y_train)
mlp_metrics, mlp_pred, mlp_proba = evaluate_model("Neural net 32-16-1", mlp, X_test, y_test)
joblib.dump(mlp, "models/churn_mlp.joblib")

all_models = pd.DataFrame([lr_metrics, rf_metrics, mlp_metrics]).set_index("model").round(3)
print("\nML vs Deep Learning on the same test set")
display(all_models)

### Was deep learning worth it?

On this tabular churn set the MLP does not clearly beat Logistic Regression or Random Forest on F1 or ROC-AUC. We only have a few thousand customers and eight features, so a 32-16 net has little extra signal to learn and takes longer to train. **For this hackathon the extra complexity is not worth it** — we keep Logistic Regression in the app.

## Task F: Review sentiment (NLP)

**What:** classify review text as Negative / Neutral / Positive.  
**Labels:** rating 1–2 Negative, 3 Neutral, 4–5 Positive.  
**Unusable rows:** drop empty / very short text (those 90 blank reviews cannot be classified from words).  
**Text prep:** strip, lowercase. **Features:** TF-IDF unigrams + bigrams. **Model:** Logistic Regression with balanced class weights because Positive reviews dominate.

In [ ]:
# Task F: TF-IDF + Logistic Regression on usable review text only.
from sklearn.feature_extraction.text import TfidfVectorizer

sent = reviews_clean.copy()
sent["review_text"] = sent["review_text"].fillna("").str.strip().str.lower()
before = len(sent)
sent = sent[sent["review_text"].str.len() >= 10].copy()
print(f"Dropped {before - len(sent)} unusable reviews (blank or too short).")

def rating_to_sentiment(rating):
    if rating <= 2:
        return "Negative"
    if rating == 3:
        return "Neutral"
    return "Positive"

sent["sentiment"] = sent["rating"].map(rating_to_sentiment)
print(sent["sentiment"].value_counts())

Xtr, Xte, ytr, yte = train_test_split(
    sent["review_text"], sent["sentiment"],
    test_size=0.20, random_state=42, stratify=sent["sentiment"],
)

sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=15000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])
sentiment_model.fit(Xtr, ytr)
yhat = sentiment_model.predict(Xte)

print(classification_report(yte, yhat, digits=3))
print("Accuracy:", round(accuracy_score(yte, yhat), 3))
print("Macro precision:", round(precision_score(yte, yhat, average="macro"), 3))
print("Macro recall:", round(recall_score(yte, yhat, average="macro"), 3))
print("Macro F1:", round(f1_score(yte, yhat, average="macro"), 3))

joblib.dump(sentiment_model, "models/sentiment_lr.joblib")
print("Saved models/sentiment_lr.joblib")

### Limitation

**Labels come from star ratings, not from the words**, and many reviews are repeated templates: the same sentence can appear with more than one rating. The model can look almost perfect because the templates line up with stars, not because it understands real customer language. In production we would need human-labeled text and unique reviews.